• Since this is an artificial scenario, you can simply assume that each active pod 
consumes 200W of electricity. As workloads run 40s on average and we have 
180 workloads in total, the expected overall energy usage is40s * 180 * 200W 
= 400 Wh. What is the carbon footprint? 

In [53]:
import pandas as pd
import re

def create_df(log_file):
    # Define regex pattern to extract relevant fields
    log_pattern = re.compile(
        r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) - INFO - Pod: (?P<pod>\S+), "
        r"Node: (?P<node>\S+), Intensity: (?P<intensity>[\d.]+), Region: (?P<region>\S+), Type: (?P<type>\S+)"
    )

    # Read and parse the log file
    data = []
    with open(log_file, "r") as file:
        for line in file:
            match = log_pattern.search(line)
            if match:
                data.append(match.groupdict())

    # Create DataFrame
    df = pd.DataFrame(data)

    # Convert columns to appropriate types
    df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime
    df["intensity"] = df["intensity"].astype(float)  # Convert intensity to float
    
    # Display DataFrame
    print(df.head())
    return df

normal_log_file = "results/normal_strategy.log"
carbon_log_file = "results/carbonaware_strategy.log"

normal = create_df(normal_log_file)
carbon = create_df(carbon_log_file)

            timestamp                  pod                  node  intensity  \
0 2025-02-01 14:48:50  workload-1738421329  normal-control-plane     254.22   
1 2025-02-01 14:48:50  workload-1738421329  normal-control-plane     254.22   
2 2025-02-01 14:49:00  workload-1738421340  normal-control-plane     252.13   
3 2025-02-01 14:49:00  workload-1738421340  normal-control-plane     252.13   
4 2025-02-01 14:49:10  workload-1738421350        normal-worker2     157.99   

  region     type  
0     DE  Planned  
1     DE   Actual  
2     DE  Planned  
3     DE   Actual  
4     NL  Planned  
            timestamp                  pod                 node  intensity  \
0 2025-02-01 14:48:48  workload-1738421327  carbonaware-worker2     160.44   
1 2025-02-01 14:48:48  workload-1738421327  carbonaware-worker2     160.44   
2 2025-02-01 14:48:58  workload-1738421338  carbonaware-worker2     160.44   
3 2025-02-01 14:48:58  workload-1738421338  carbonaware-worker2     160.44   
4 2025-02-01 14

C:\Users\morit\AppData\Local\Temp\ipykernel_18316\1441992168.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime
C:\Users\morit\AppData\Local\Temp\ipykernel_18316\1441992168.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"])  # Convert timestamp to datetime


Normal

In [54]:
def analyze_df(df):
    df = df.loc[df['type'] == "Actual"]
    
    # Calculate average intensity overall and per node
    avg_intensity_overall = df["intensity"].mean()
    avg_intensity_per_node = df.groupby("node")["intensity"].mean()
    workload_count_per_node = df["node"].value_counts(normalize=True) * 100
    avg_footprint = avg_intensity_overall * 0.4

    # Display results
    print("Average Intensity Overall:", avg_intensity_overall)
    print("Average Footprint Overall:", avg_footprint)
    print("Average Intensity Per Node:", avg_intensity_per_node, end= "\n\n")
    print("workload_count_per_node" , workload_count_per_node, end= "\n\n")


In [55]:
analyze_df(normal)

Average Intensity Overall: 290.7154444444444
Average Footprint Overall: 116.28617777777777
Average Intensity Per Node: node
normal-control-plane    333.999692
normal-worker           311.118654
normal-worker2          229.216349
Name: intensity, dtype: float64

workload_count_per_node node
normal-control-plane    36.111111
normal-worker2          35.000000
normal-worker           28.888889
Name: proportion, dtype: float64



Carbon

In [56]:
analyze_df(carbon)

Average Intensity Overall: 212.74977777777772
Average Footprint Overall: 85.0999111111111
Average Intensity Per Node: node
carbonaware-worker     265.671897
carbonaware-worker2    187.590082
Name: intensity, dtype: float64

workload_count_per_node node
carbonaware-worker2    67.777778
carbonaware-worker     32.222222
Name: proportion, dtype: float64



In [58]:
carbon = carbon.loc[carbon['type'] == "Actual"]
normal = normal.loc[normal['type'] == "Actual"]

carbon_fp=carbon["intensity"].mean()*0.4
normal_fp=normal["intensity"].mean()*0.4

(normal_fp-carbon_fp)/normal_fp


0.26818549945173575